## 1. Setup & configuration

In [ ]:
import os, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.feature_selection import mutual_info_classif
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, matthews_corrcoef, cohen_kappa_score
from imblearn.over_sampling import SMOTE

import warnings; warnings.filterwarnings("ignore")

SEED = 42
RNG = np.random.default_rng(SEED)
torch.manual_seed(SEED)

# ---- locate the uploaded CSVs (same folder as the notebook, else the upload dir) ----
def _find(name):
    for d in [".", "/mnt/user-data/uploads", "/mnt/user-data/outputs"]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Could not find {name}; place it next to the notebook.")
CCSR_PATH = _find("code_to_CCSR.csv")
DATA_PATH = _find("clinical_notes_dataset.csv")
VITALS_PATH = _find("vitals.csv")   # long-format vitals export (required upload)

# ---- feature/config switches ----
CLINICAL_BERT_MODEL    = "medicalai/ClinicalBERT"   # diagnostic & procedure text are embedded with this
HAS_VITALS             = True    # add the LSTM-autoencoder vital block 
DEMOGRAPHICS           = "file"   # gender & ethnicity read from dataset columns (assumed present)
TEXT_EMB_DIM           = 768     # Clinical BERT embedding size per text field
N_PHEN                 = 12       # phenotype topics per target, as in the paper
EPOCHS                 = 100       # contrastive-training epochs 
print("CCSR map:", CCSR_PATH, "| dataset:", DATA_PATH, "| vitals:", VITALS_PATH)

## 2. Dataset preparation 

### 2.1 Map diagnosis codes to CCSR categories and assemble per-encounter records

Diagnoses are constant within an encounter, so they are read once per encounter. An encounter can
have **several procedure reports** (multiple `redcap_repeat_instance` rows); all of them are
concatenated into a single procedure text, which is embedded once in 2.3.

In [ ]:
# --- CCSR map: ICD-10 code -> list of clinical category names (user-specified logic) ---
ccsr = pd.read_csv(CCSR_PATH)
ccsr.columns = [c.strip() for c in ccsr.columns]
ccsr["Code"] = ccsr["Code"].astype(str).str.strip()
ccsr["Category"] = ccsr["Category"].astype(str).str.strip()
code2cats = ccsr.groupby("Code")["Category"].apply(list).to_dict()
print(f"CCSR: {ccsr.Code.nunique()} codes -> {ccsr.Category.nunique()} categories")

HF_CATEGORY = "Heart failure"

def strip_decimal(code):
    return str(code).replace(".", "").strip().upper()

def split_codes(val):
    if pd.isna(val): return []
    return [c.strip() for c in re.split(r"[;,|/]+|\s{2,}", str(val)) if c.strip()]

def codes_to_text(codes, drop=frozenset([HF_CATEGORY])):
    cats = []
    for c in codes:
        for cat in code2cats.get(strip_decimal(c), []):
            if cat not in drop:
                cats.append(cat)
    return "; ".join(cats)

def codes_to_cats(codes, drop=frozenset([HF_CATEGORY])):
    """Same mapping as codes_to_text, but returns a de-duplicated list for phenotype counts."""
    cats = []
    for c in codes:
        for cat in code2cats.get(strip_decimal(c), []):
            if cat not in drop and cat not in cats:
                cats.append(cat)
    return cats

def extract_impression(text):
    """Paper rule: take Impression/Conclusion; if absent use Findings; else the whole note."""
    text = str(text)
    for key in ("IMPRESSION", "CONCLUSION"):
        m = re.search(key + r"\s*:?\s*(.*)", text, re.IGNORECASE | re.DOTALL)
        if m and m.group(1).strip():
            return m.group(1).strip()
    m = re.search(r"FINDINGS\s*:?\s*(.*)", text, re.IGNORECASE | re.DOTALL)
    return m.group(1).strip() if m else text

# --- group rows by hospitalization (encounter_id); combine primary + secondary codes ---
df = pd.read_csv(DATA_PATH)
df["Admission Date"] = pd.to_datetime(df["Admission Date"])
df["Discharge Date"] = pd.to_datetime(df["Discharge Date"])

records = []
for enc_id, grp in df.groupby("encounter_id"):
    grp = grp.sort_values("redcap_repeat_instance") if "redcap_repeat_instance" in grp else grp
    # diagnoses are constant within an encounter -> read them from the first row
    codes = split_codes(grp["primary diagnosis"].iloc[0]) + split_codes(grp["secondary diagnosis"].iloc[0])
    cats  = codes_to_cats(codes)                 # de-duplicated category list (Heart failure dropped)
    # an encounter may have SEVERAL procedure reports -> concatenate them all, then embed once
    reports = [extract_impression(t) for t in grp["procedure report"].tolist()
               if str(t).strip() and str(t).lower() != "nan"]
    proc_text = re.sub(r"\s+", " ", " ".join(reports)).strip()
    los_days = (grp["Discharge Date"].iloc[0] - grp["Admission Date"].iloc[0]).days
    records.append({
        "encounter_id": enc_id,
        "patient_id":   grp["record_id"].iloc[0],
        "diag_cats":    cats,
        "diag_text":    codes_to_text(codes),    # user-specified text rendering
        "proc_text":    proc_text,               # all procedure reports for the encounter, concatenated
        "n_proc":       len(reports),
        "los_days":     los_days,
    })

E = pd.DataFrame(records).reset_index(drop=True)
E = E[E["los_days"] >= 0].reset_index(drop=True)        # keep valid stays (discharge >= admission)
# completeness rule: every encounter must have diagnostic categories AND a procedure report
# (vitals are assumed present for every encounter; see 2.4b)
E["proc_text"] = E["proc_text"].fillna("").astype(str)
_complete = E["diag_cats"].map(len).gt(0) & E["proc_text"].str.strip().ne("")
print(f"dropping {int((~_complete).sum())} encounters missing diagnoses/procedures")
E = E[_complete].reset_index(drop=True)
N = len(E)
print(f"{N} hospitalizations from {E['patient_id'].nunique()} patients")
print(f"procedure reports per encounter: mean {E['n_proc'].mean():.1f}, max {int(E['n_proc'].max())} "
      f"(all concatenated before embedding)")
E[["encounter_id", "diag_text", "proc_text", "n_proc", "los_days"]].head(3)

### 2.2 Length of stay → 5 categories 

In [ ]:
# Length of stay: days from the encounter dates, then 5 ordinal classes
#   class 0 very short-term  0-1 d      class 3 long-term       15-21 d
#   class 1 short-term       2-7 d      class 4 very long-term  >21 d
#   class 2 medium-term      8-14 d
def los_category(d):
    if d <= 1:  return 0
    if d <= 7:  return 1
    if d <= 14: return 2
    if d <= 21: return 3
    return 4

LOS_NAMES = ["very short (0-1d)", "short (2-7d)", "medium (8-14d)",
             "long (15-21d)", "very long (>21d)"]
E["los_class"] = E["los_days"].apply(los_category)
los = E["los_class"].values
counts = E["los_class"].value_counts().sort_index()
print("Length-of-stay distribution:")
for i, name in enumerate(LOS_NAMES):
    print(f"  class {i}  {name:<18}: {int(counts.get(i, 0))}")

### 2.3 Text embeddings — Clinical BERT 

The diagnostic-code text and the procedure-report text are each embedded with
`medicalai/ClinicalBERT` (768-d, mean-pooled over tokens). 

In [ ]:
def clinical_bert_embed(texts, batch_size=32):
    """Mean-pooled Clinical BERT embeddings (768-d). Loads the model once and caches it."""
    from transformers import AutoTokenizer, AutoModel
    if not hasattr(clinical_bert_embed, "_model"):
        clinical_bert_embed._tok   = AutoTokenizer.from_pretrained(CLINICAL_BERT_MODEL)
        clinical_bert_embed._model = AutoModel.from_pretrained(CLINICAL_BERT_MODEL).eval()
    tok, model = clinical_bert_embed._tok, clinical_bert_embed._model
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = [str(t) for t in texts[i:i+batch_size]]
            enc = tok(batch, return_tensors="pt", truncation=True, max_length=512, padding=True)
            h = model(**enc).last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1).float()
            emb = (h * mask).sum(1) / mask.sum(1).clamp(min=1)      # mean-pool over real tokens
            out.append(emb.cpu().numpy())
    return np.concatenate(out, 0).astype(np.float32)

diag_embeddings = clinical_bert_embed(E["diag_text"].tolist())
proc_embeddings = clinical_bert_embed(E["proc_text"].tolist())
print("Diagnostic embeddings:", diag_embeddings.shape, "| Procedure embeddings:", proc_embeddings.shape)

### 2.4 Clinical phenotype topics (targets for SC-WEAT)

The phenotypes are the **topics** from our previous study (https://ieeexplore.ieee.org/abstract/document/10224690). They are stored as
per-encounter **names** in the dataset columns `diag_topic` and `proc_topic`; this cell reads
those names and maps each to its **code 1–12** (position in the lists below). The per-encounter
one-hot membership is the SC-WEAT target.

* Diagnostic-code topics (1–12): Ischemic Heart Disease, Renal Dysfunction, Hypertensive Heart
  Disease, Gastrointestinal Disease, Skills and Self-Care Behaviors, Cardiovascular Diseases, Acute
  Heart Failure, Heart Failure Comorbidities, Chronic Heart Disease, Chronic Obstructive Pulmonary
  Disease, Heart Failure Disorders, Non-Ischemic Heart Disease.
* Procedure-report topics (1–12): Ischemic Heart Disease, Effusions, Hypertrophic Cardiomyopathy,
  Arrhythmia, Idiopathic Cardiomyopathy, Common Findings, Myocardial Infarction, Atrioventricular
  Septal Defect, Congestion and Heart Block, Cardiopulmonary Disease, Left Ventricular Dysfunction,
  Myocardial Ischemia.

In [ ]:
DIAG_TOPICS = ["Ischemic Heart Disease", "Renal Dysfunction", "Hypertensive Heart Disease",
               "Gastrointestinal Disease", "Skills and Self-Care Behaviors", "Cardiovascular Diseases",
               "Acute Heart Failure", "Heart Failure Comorbidities", "Chronic Heart Disease",
               "Chronic Obstructive Pulmonary Disease", "Heart Failure Disorders", "Non-Ischemic Heart Disease"]
PROC_TOPICS = ["Ischemic Heart Disease", "Effusions", "Hypertrophic Cardiomyopathy", "Arrhythmia",
               "Idiopathic Cardiomyopathy", "Common Findings", "Myocardial Infarction",
               "Atrioventricular Septal Defect", "Congestion and Heart Block", "Cardiopulmonary Disease",
               "Left Ventricular Dysfunction", "Myocardial Ischemia"]
DIAG_CODE = {name: i + 1 for i, name in enumerate(DIAG_TOPICS)}   # topic name -> code 1..12
PROC_CODE = {name: i + 1 for i, name in enumerate(PROC_TOPICS)}

# read the topic NAMES from the dataset (one per encounter) and map to codes 1..12
diag_name = df.groupby("encounter_id")["diag_topic"].first().reindex(E["encounter_id"]).astype(str).str.strip()
proc_name = df.groupby("encounter_id")["proc_topic"].first().reindex(E["encounter_id"]).astype(str).str.strip()
diag_code = diag_name.map(DIAG_CODE).astype(int).values                  # 1..12
proc_code = proc_name.map(PROC_CODE).astype(int).values
E["diag_topic_name"], E["diag_topic"] = diag_name.values, diag_code
E["proc_topic_name"], E["proc_topic"] = proc_name.values, proc_code

# one-hot phenotype targets (N x 12) used by SC-WEAT
diag_phenotypes = np.zeros((N, 12), dtype=int); diag_phenotypes[np.arange(N), diag_code - 1] = 1
proc_phenotypes = np.zeros((N, 12), dtype=int); proc_phenotypes[np.arange(N), proc_code - 1] = 1
DIAG_PHENOTYPES, PROC_PHENOTYPES = DIAG_TOPICS, PROC_TOPICS
DIAG_IDX = {c: i for i, c in enumerate(DIAG_TOPICS)}; PHEN_IDX = DIAG_IDX   # back-compat alias

print("Diagnostic topic counts (codes 1..12):", np.bincount(diag_code, minlength=13)[1:].tolist())
print("Procedure  topic counts (codes 1..12):", np.bincount(proc_code, minlength=13)[1:].tolist())
E[["encounter_id", "diag_topic_name", "diag_topic", "proc_topic_name", "proc_topic"]].head()

### 2.4b Physiological-vital embeddings (LSTM autoencoder)

Vitals come from **`vitals.csv`** (a required upload; long format: one row per reading,
`VITAL_NAME / VITAL_DATE / VITAL_VALUE`) and are **assumed present for every encounter**. The number
of readings varies per encounter and per vital — some have many, some only one. Six vitals are kept
(pulse, systolic/diastolic BP, respiratory rate, temperature °C, SpO₂); any out-of-range value is
dropped, each vital is ordered by time and resampled to a fixed-length sequence (a single reading is
repeated, many are interpolated), and a two-layer **LSTM autoencoder** encodes each to 100-d
(6 × 100 = 600-d).

In [ ]:
N_VITALS, VITAL_DIM, SEQ_LEN = 6, 100, 24

class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features=1, latent=VITAL_DIM):
        super().__init__()
        self.encoder = nn.LSTM(n_features, latent, num_layers=2, batch_first=True)
        self.decoder = nn.LSTM(latent, latent, num_layers=2, batch_first=True)
        self.out = nn.Linear(latent, n_features)
    def forward(self, x):
        _, (h, _) = self.encoder(x)
        z = torch.relu(h[-1])
        dec, _ = self.decoder(z.unsqueeze(1).repeat(1, x.size(1), 1))
        return self.out(dec), z

def embed_vital(seq_1d, epochs=5, bs=256):
    x = torch.tensor(seq_1d, dtype=torch.float32).unsqueeze(-1)
    model = LSTMAutoencoder(); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(epochs):
        perm = torch.randperm(x.size(0))
        for i in range(0, x.size(0), bs):
            idx = perm[i:i+bs]; opt.zero_grad()
            recon, _ = model(x[idx]); F.mse_loss(recon, x[idx]).backward(); opt.step()
    with torch.no_grad():
        _, z = model(x)
    return z.numpy()

# --- load vitals (assumed present for every encounter) and build per-encounter sequences ---
CANON = {"Pulse Rate":"heart_rate", "BP Systolic":"systolic_bp", "BP Diastolic":"diastolic_bp",
         "Respiratory Rate":"resp_rate", "Temp (DegC)":"temp_C", "SPO2":"spo2"}
VITAL_ORDER = ["heart_rate","systolic_bp","diastolic_bp","resp_rate","temp_C","spo2"]
RANGES = {"heart_rate":(30,200), "systolic_bp":(60,260), "diastolic_bp":(30,160),
          "resp_rate":(5,60), "temp_C":(33,43), "spo2":(50,100)}

def _resample(seq, n=SEQ_LEN):
    seq = np.asarray(seq, dtype=float)
    if len(seq) == 0: return None
    if len(seq) == 1: return np.repeat(seq, n)               # a single reading -> repeated
    return np.interp(np.linspace(0, 1, n), np.linspace(0, 1, len(seq)), seq)  # many -> interpolated

vit = pd.read_csv(VITALS_PATH); vit.columns = [c.strip() for c in vit.columns]
vit["canon"] = vit["VITAL_NAME"].map(CANON)
vit["val"]   = pd.to_numeric(vit["VITAL_VALUE"], errors="coerce")
vit["dt"]    = pd.to_datetime(vit["VITAL_DATE"], errors="coerce")
vit = vit[vit["canon"].notna() & vit["val"].notna() & vit["dt"].notna()]
vit = vit[vit.apply(lambda r: RANGES[r["canon"]][0] <= r["val"] <= RANGES[r["canon"]][1], axis=1)]
print(f"clean canonical vital readings: {len(vit)} over {vit.ENCOUNTER_ID.nunique()} encounters")

# every encounter has all six vitals -> build sequences directly
groups = {int(enc): g for enc, g in vit.groupby("ENCOUNTER_ID")}
vital_sequences = np.zeros((N, N_VITALS, SEQ_LEN), dtype=np.float32)
for i, enc in enumerate(E["encounter_id"].astype(int).values):
    g = groups.get(enc)
    for vi, v in enumerate(VITAL_ORDER):
        vals = g[g["canon"] == v].sort_values("dt")["val"].values
        vital_sequences[i, vi, :] = _resample(vals)
print(f"built vital sequences for all {N} encounters (6 vitals each)")

vs = vital_sequences.copy()                                  # z-score each vital for the AE
for vi in range(N_VITALS):
    m, s = vs[:, vi, :].mean(), vs[:, vi, :].std() + 1e-9
    vs[:, vi, :] = (vs[:, vi, :] - m) / s
vital_embeddings = np.concatenate([embed_vital(vs[:, v, :]) for v in range(N_VITALS)],
                                  axis=1).astype(np.float32)
print("Vital embeddings:", vital_embeddings.shape, "(6 vitals x 100-d)")

### 2.5 Demographic attributes (gender & ethnicity)

`gender` and `ethnicity` are columns in `clinical_notes_dataset.csv` (one value per encounter, gender
~balanced, ethnicity imbalanced). This cell reads them straight from the dataset and binarises them:
gender F→0 / M→1, ethnicity Hispanic→0 / Non-Hispanic→1; ethnicity's imbalance is handled with SMOTE
downstream.

In [ ]:
def _col_per_encounter(col):
    return E["encounter_id"].map(df.groupby("encounter_id")[col].first())

# gender and ethnicity are present per encounter in the dataset -> read and binarise
g = _col_per_encounter("gender").astype(str).str.strip().str.lower()
e = _col_per_encounter("ethnicity").astype(str).str.strip().str.lower()
gender    = g.map({"female": 0, "f": 0, "male": 1, "m": 1}).astype(int).values
ethnicity = e.map({"hispanic": 0, "non-hispanic": 1, "nonhispanic": 1}).astype(int).values

print("Demographics read from dataset columns 'gender' and 'ethnicity'.")
print("Gender    (0=F,1=M):       ", np.bincount(gender),    "(fairly balanced)")
print("Ethnicity (0=Hisp,1=nonH): ", np.bincount(ethnicity), "(imbalanced -> SMOTE)")

### 2.6 Assemble the feature matrix

In [ ]:
blocks = [diag_embeddings, proc_embeddings] + ([vital_embeddings] if HAS_VITALS else [])
X = np.concatenate(blocks, axis=1).astype(np.float32)
FEAT_DIM = X.shape[1]
print("Feature matrix X:", X.shape, f"({'with' if HAS_VITALS else 'no'} vitals)")

### 2.7 Dataset after embeddings

A per-encounter view that pairs the metadata (record id, encounter, length of stay, gender,
ethnicity, diagnostic/procedure topics) with the learned embeddings (diagnostic 768-d, procedure
768-d, vitals 600-d). The embedding columns are named `diag_0…767`, `proc_0…767`, `vit_0…599`; the
table is also written to `dataset_after_embeddings.csv`.

In [ ]:
meta = pd.DataFrame({
    "record_id":      E["patient_id"].values,
    "encounter_id":   E["encounter_id"].values,
    "los_days":       E["los_days"].values,
    "los_class":      los,
    "gender":         gender,
    "ethnicity":      ethnicity,
    "diag_topic":     E["diag_topic"].values,        # code 1..12
    "proc_topic":     E["proc_topic"].values,        # code 1..12
    "diag_topic_name": E["diag_topic_name"].values,
    "proc_topic_name": E["proc_topic_name"].values,
})
diag_cols = pd.DataFrame(diag_embeddings, columns=[f"diag_{i}" for i in range(diag_embeddings.shape[1])])
proc_cols = pd.DataFrame(proc_embeddings, columns=[f"proc_{i}" for i in range(proc_embeddings.shape[1])])
vit_cols  = (pd.DataFrame(vital_embeddings, columns=[f"vit_{i}" for i in range(vital_embeddings.shape[1])])
             if HAS_VITALS else pd.DataFrame(index=range(N)))

dataset_after_embeddings = pd.concat([meta, diag_cols, proc_cols, vit_cols], axis=1)
dataset_after_embeddings.to_csv("dataset_after_embeddings.csv", index=False)

print("dataset_after_embeddings:", dataset_after_embeddings.shape,
      f"= {meta.shape[1]} metadata + {diag_embeddings.shape[1]} diag + "
      f"{proc_embeddings.shape[1]} proc + {vital_embeddings.shape[1] if HAS_VITALS else 0} vitals")
# show ALL columns (metadata + every diag/proc/vital embedding dimension)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
dataset_after_embeddings.head()

## 3. Feature selection for the sensitive attribute 

Top-50% of features by **mutual information** with the sensitive attribute are kept as *sensitive
features*; after **SMOTE** balancing, five classifiers confirm they predict the attribute.

In [ ]:
def smote_balance(X, y):
    """SMOTE to balance classes (used for the imbalanced ethnicity attribute). k adapts to the
    minority size so it is safe on small cohorts; near-balanced inputs are returned unchanged."""
    cnt = np.bincount(y)
    if cnt.min() / cnt.max() >= 0.8 or cnt.min() <= 1:
        return X, y
    k = int(min(5, cnt.min() - 1))
    return SMOTE(random_state=SEED, k_neighbors=k).fit_resample(X, y)

def select_sensitive_features(X, sensitive, fraction=0.5):
    mi = mutual_info_classif(X, sensitive, random_state=SEED)
    k = int(len(mi) * fraction)
    return np.argsort(mi)[::-1][:k]

def predict_sensitive_attribute(X, sensitive, feat_idx=None):
    feats = X if feat_idx is None else X[:, feat_idx]
    Xb, yb = smote_balance(feats, sensitive)                    # balance ethnicity before evaluating
    Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.2, random_state=SEED, stratify=yb)
    models = {
        "kNN": KNeighborsClassifier(),
        "LR":  make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)),
        "SVM": make_pipeline(StandardScaler(), SVC()),
        "MLP": make_pipeline(StandardScaler(), MLPClassifier(max_iter=200, hidden_layer_sizes=(64,))),
        "RF":  RandomForestClassifier(n_estimators=100, random_state=SEED),
    }
    return {n: round(accuracy_score(yte, m.fit(Xtr, ytr).predict(Xte)), 3) for n, m in models.items()}

rows = []
for name, sens in [("Gender", gender), ("Ethnicity", ethnicity)]:
    sf = select_sensitive_features(X, sens)
    rows.append({"Sensitive attribute": name, "Features": "All",       **predict_sensitive_attribute(X, sens)})
    rows.append({"Sensitive attribute": name, "Features": "Sensitive", **predict_sensitive_attribute(X, sens, sf)})
table_I = pd.DataFrame(rows).set_index(["Sensitive attribute", "Features"])
print("TABLE I - Prediction accuracy of the sensitive attribute")
table_I

## 4. Counterfactual example generation 

For each record a **positive** is built by replacing its sensitive features with the opposite
class's mean sensitive-feature vector; all other records are negatives.

In [ ]:
def generate_counterfactual_positives(X, sensitive, feat_idx):
    mean_class0 = X[sensitive == 0][:, feat_idx].mean(axis=0)
    mean_class1 = X[sensitive == 1][:, feat_idx].mean(axis=0)
    pos = X.copy()
    pos[np.ix_(np.where(sensitive == 0)[0], feat_idx)] = mean_class1
    pos[np.ix_(np.where(sensitive == 1)[0], feat_idx)] = mean_class0
    return pos.astype(np.float32)

SF_FULL = {"gender": select_sensitive_features(X, gender),
           "ethnicity": select_sensitive_features(X, ethnicity)}
print("Sensitive features per attribute:", {k: len(v) for k, v in SF_FULL.items()})

## 5. The Debias-CLR framework 

An encoder trained with the **NT-Xent** loss (Eq. 1) and the **LARS** optimiser pulls each record
toward its counterfactual positive and away from negatives. The paper uses 100 epochs / batch
1024; smaller defaults keep this fast on 256 records.

In [ ]:
class DebiasEncoder(nn.Module):
    def __init__(self, d=None, hidden=256):
        super().__init__()
        d = d or FEAT_DIM
        self.net = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Linear(hidden, d))
    def forward(self, x):
        return x + self.net(x)   # residual: keep information, learn only the debiasing delta

def nt_xent_loss(z1, z2, temperature=0.5):
    z1, z2 = F.normalize(z1, dim=1), F.normalize(z2, dim=1)
    n = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    sim = (z @ z.T) / temperature
    sim.fill_diagonal_(-9e15)
    targets = torch.cat([torch.arange(n) + n, torch.arange(n)]).to(z.device)
    return F.cross_entropy(sim, targets)

class LARS(torch.optim.Optimizer):
    def __init__(self, params, lr=0.1, momentum=0.9, weight_decay=1e-4, eta=1e-3):
        super().__init__(params, dict(lr=lr, momentum=momentum, weight_decay=weight_decay, eta=eta))
    @torch.no_grad()
    def step(self):
        for grp in self.param_groups:
            for p in grp["params"]:
                if p.grad is None:
                    continue
                d_p = p.grad.add(p, alpha=grp["weight_decay"])
                p_norm, g_norm = p.norm(), d_p.norm()
                local_lr = (grp["eta"] * p_norm / (g_norm + 1e-9)) if (p_norm > 0 and g_norm > 0) else 1.0
                buf = self.state[p].setdefault("momentum_buffer", torch.zeros_like(p))
                buf.mul_(grp["momentum"]).add_(d_p, alpha=grp["lr"] * float(local_lr))
                p.add_(buf, alpha=-1.0)

def train_debias_clr(X, sensitive, feat_idx, epochs=60, batch_size=128, verbose=False):
    # SMOTE-balance the sensitive groups (e.g. imbalanced ethnicity) for training only
    Xb, sb = smote_balance(X, sensitive)
    positives = generate_counterfactual_positives(Xb, sb, feat_idx)
    enc = DebiasEncoder(X.shape[1]); opt = LARS(enc.parameters(), lr=0.05)
    Xt, Pt = torch.tensor(Xb), torch.tensor(positives); n = Xt.shape[0]
    for ep in range(epochs):
        perm = torch.randperm(n); total = 0.0
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]; opt.zero_grad()
            loss = nt_xent_loss(enc(Xt[idx]), enc(Pt[idx])); loss.backward(); opt.step(); total += loss.item()
        if verbose and (ep % 20 == 0 or ep == epochs - 1):
            print(f"  epoch {ep:>2}  loss {total:.3f}")
    enc.eval()
    with torch.no_grad():                                        # embed the ORIGINAL encounters
        return enc(torch.tensor(X)).numpy().astype(np.float32)

_ = train_debias_clr(X, gender, SF_FULL["gender"], epochs=10, verbose=True)
print("Debias-CLR trains OK.")

## 6. SC-WEAT fairness metric 

Each phenotype target is represented by its **prototype** (mean embedding of records carrying it);
the effect size compares how the two classes associate with those prototypes. 0 ⇒ fair.

In [ ]:
def _cosine(A, B):
    An = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-9)
    Bn = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-9)
    return An @ Bn.T

def sc_weat_effect_size(embeddings, sensitive, phenotype_onehot):
    prototypes = np.array([embeddings[phenotype_onehot[:, j] == 1].mean(axis=0)
                           for j in range(phenotype_onehot.shape[1])
                           if phenotype_onehot[:, j].sum() > 0])
    s1 = _cosine(prototypes, embeddings[sensitive == 0]).ravel()
    s2 = _cosine(prototypes, embeddings[sensitive == 1]).ravel()
    return float((s1.mean() - s2.mean()) / (np.concatenate([s1, s2]).std() + 1e-9))

print("SC-WEAT (gender, diagnostic phenotypes), raw embeddings:",
      round(sc_weat_effect_size(X, gender, diag_phenotypes), 3))

## 7. Feature regularisation — Debias-CLR-R 

Cutout: zero a random 20% of each record's sensitive features before training.

In [ ]:
def cutout_sensitive_features(X, feat_idx, rate=0.2):
    Xr = X.copy(); n_mask = int(len(feat_idx) * rate)
    for i in range(X.shape[0]):
        Xr[i, RNG.choice(feat_idx, n_mask, replace=False)] = 0.0
    return Xr.astype(np.float32)

def train_debias_clr_r(X, sensitive, feat_idx, **kw):
    return train_debias_clr(cutout_sensitive_features(X, feat_idx), sensitive, feat_idx, **kw)
print("Debias-CLR-R ready.")

## 7b. Train the two Debias-CLR models (one per sensitive attribute)

Two **independent** encoders are trained — one to debias **gender**, one to debias **ethnicity** —
each using its own mutual-information-selected sensitive features. Their output embeddings (and the
Debias-CLR-R cutout variants) are stored once and reused for the fairness sweep and the
length-of-stay evaluation below.

In [ ]:
print("Training Debias-CLR model 1/2  (gender) ...")
emb_gender      = train_debias_clr(X,   gender,    SF_FULL["gender"],    epochs=EPOCHS)
emb_gender_r    = train_debias_clr_r(X, gender,    SF_FULL["gender"],    epochs=EPOCHS)
print("Training Debias-CLR model 2/2  (ethnicity) ...")
emb_ethnicity   = train_debias_clr(X,   ethnicity, SF_FULL["ethnicity"], epochs=EPOCHS)
emb_ethnicity_r = train_debias_clr_r(X, ethnicity, SF_FULL["ethnicity"], epochs=EPOCHS)

DEBIAS = {"gender":    {"Debias-CLR": emb_gender,    "Debias-CLR-R": emb_gender_r},
          "ethnicity": {"Debias-CLR": emb_ethnicity, "Debias-CLR-R": emb_ethnicity_r}}

for lab in ("gender", "ethnicity"):
    sens = gender if lab == "gender" else ethnicity
    before = sc_weat_effect_size(X, sens, diag_phenotypes)
    after  = sc_weat_effect_size(DEBIAS[lab]["Debias-CLR"], sens, diag_phenotypes)
    print(f"  {lab:>9}: SC-WEAT (diagnostic) {before:+.2f} -> {after:+.2f} after Debias-CLR")

## 8. SC-WEAT effect size vs training-data size 

Effect size before debiasing, after Debias-CLR, and after Debias-CLR-R, for each attribute,
target and training fraction. Raise `EPOCHS` toward 100 for stronger debiasing.

In [ ]:
FRACTIONS = [0.2, 0.4, 0.6, 0.8]

def run_sweep(sensitive, label):
    out = []
    for frac in FRACTIONS:
        n = max(40, int(N * frac))
        sel = RNG.choice(N, n, replace=False)
        Xs, ss = X[sel], sensitive[sel]
        dphen, pphen = diag_phenotypes[sel], proc_phenotypes[sel]
        fi = select_sensitive_features(Xs, ss)
        deb   = train_debias_clr(Xs, ss, fi, epochs=EPOCHS)
        deb_r = train_debias_clr_r(Xs, ss, fi, epochs=EPOCHS)
        for target, phen in [("Diagnostic codes", dphen), ("Procedure reports", pphen)]:
            out.append({"Sensitive": label, "Target": target, "% data": int(frac*100),
                        "Before":       round(sc_weat_effect_size(Xs,    ss, phen), 1),
                        "Debias-CLR":   round(sc_weat_effect_size(deb,   ss, phen), 1),
                        "Debias-CLR-R": round(sc_weat_effect_size(deb_r, ss, phen), 1)})
    return out

table_II = pd.DataFrame(run_sweep(gender, "Gender") + run_sweep(ethnicity, "Ethnicity")
                        ).set_index(["Sensitive", "Target", "% data"])
print("TABLE II - SC-WEAT effect sizes vs % of training data")
table_II

## 9. Length-of-stay prediction 

Length of stay is binarised (very-short+short vs the rest), SMOTE-balanced, and predicted from
raw / Debias-CLR / Debias-CLR-R embeddings with Accuracy, MCC and Cohen's Kappa.

In [ ]:
los_binary = (los >= 2).astype(int)

def evaluate_los(embeddings, y):
    Xb, yb = SMOTE(random_state=SEED).fit_resample(embeddings, y)
    Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.2, random_state=SEED, stratify=yb)
    models = {
        "kNN": KNeighborsClassifier(),
        "LR":  make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)),
        "SVM": make_pipeline(StandardScaler(), SVC()),
        "MLP": make_pipeline(StandardScaler(), MLPClassifier(max_iter=200, hidden_layer_sizes=(64,))),
        "RF":  RandomForestClassifier(n_estimators=100, random_state=SEED),
    }
    res = {}
    for name, m in models.items():
        pred = m.fit(Xtr, ytr).predict(Xte)
        res[name] = (round(accuracy_score(yte, pred), 3),
                     round(matthews_corrcoef(yte, pred), 3),
                     round(cohen_kappa_score(yte, pred), 3))
    return res

def los_table(label):
    embeds = {"Raw Embeddings": X, **DEBIAS[label]}   # reuse the two models trained in 7b
    rows = []
    for model in ["kNN", "LR", "SVM", "MLP", "RF"]:
        for emb_name, emb in embeds.items():
            a, mcc, k = evaluate_los(emb, los_binary)[model]
            rows.append({"Model": model, "Embeddings": emb_name, "A": a, "MCC": mcc, "K": k})
    print(f"\nTABLE - Length-of-stay prediction ({label} debiasing)")
    return pd.DataFrame(rows).set_index(["Model", "Embeddings"])

table_III = los_table("gender")
table_III

In [ ]:
table_IV = los_table("ethnicity")
table_IV